In [1]:
import kagglehub

path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")

print("Path to dataset files:", path)

c:\Users\DBeye\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\DBeye\.cache\kagglehub\datasets\maharshipandya\-spotify-tracks-dataset\versions\1


In [ ]:
import pandas as pd
import numpy as np
import os

path = "/Users/zzzhao/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1"

csv_file = os.path.join(path, "dataset.csv") 
df = pd.read_csv(csv_file)

print(df.head())


   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [3]:
df = df.drop(columns=["Unnamed: 0", "track_id", "album_name"])
df["explicit"] = df["explicit"].astype(int)
df = df.dropna()
print(df.columns)
print(df["track_genre"].unique())  


Index(['artists', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'track_genre'],
      dtype='object')
['acoustic' 'afrobeat' 'alt-rock' 'alternative' 'ambient' 'anime'
 'black-metal' 'bluegrass' 'blues' 'brazil' 'breakbeat' 'british'
 'cantopop' 'chicago-house' 'children' 'chill' 'classical' 'club' 'comedy'
 'country' 'dance' 'dancehall' 'death-metal' 'deep-house' 'detroit-techno'
 'disco' 'disney' 'drum-and-bass' 'dub' 'dubstep' 'edm' 'electro'
 'electronic' 'emo' 'folk' 'forro' 'french' 'funk' 'garage' 'german'
 'gospel' 'goth' 'grindcore' 'groove' 'grunge' 'guitar' 'happy'
 'hard-rock' 'hardcore' 'hardstyle' 'heavy-metal' 'hip-hop' 'honky-tonk'
 'house' 'idm' 'indian' 'indie-pop' 'indie' 'industrial' 'iranian'
 'j-dance' 'j-idol' 'j-pop' 'j-rock' 'jazz' 'k-pop' 'kids' 'latin'
 'latino' 'malay' 'mandopo

In [4]:
from sklearn.preprocessing import LabelEncoder

full_genre_list = [
    'acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime',
    'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british',
    'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club',
    'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house',
    'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep',
    'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk',
    'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge',
    'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal',
    'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie-pop', 'indie',
    'industrial', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'jazz',
    'k-pop', 'kids', 'latin', 'latino', 'malay', 'mandopop', 'metal',
    'metalcore', 'minimal-techno', 'mpb', 'new-age', 'opera', 'pagode',
    'party', 'piano', 'pop-film', 'pop', 'power-pop', 'progressive-house',
    'psych-rock', 'punk-rock', 'punk', 'r-n-b', 'reggae', 'reggaeton',
    'rock-n-roll', 'rock', 'rockabilly', 'romance', 'sad', 'salsa', 'samba',
    'sertanejo', 'show-tunes', 'singer-songwriter', 'ska', 'sleep',
    'songwriter', 'soul', 'spanish', 'study', 'swedish', 'synth-pop', 'tango',
    'techno', 'trance', 'trip-hop', 'turkish', 'world-music'
]

df['genre'] = df['track_genre'].str.strip().str.lower()

valid_genres = set(full_genre_list)

df['is_valid_genre'] = df['genre'].isin(valid_genres)

df['genre'] = df.apply(
    lambda x: x['genre'] if x['is_valid_genre'] else 'unknown',
    axis=1
)


full_genre_list_with_unknown = ['unknown'] + full_genre_list

label_encoder = LabelEncoder()
label_encoder.fit(full_genre_list_with_unknown)  

df['genre_code'] = label_encoder.transform(df['genre'])

genre_mapping = pd.DataFrame({
    'genre_name': label_encoder.classes_,
    'genre_code': label_encoder.transform(label_encoder.classes_)
}).sort_values('genre_code')

df = df.drop(columns=['is_valid_genre'])

print(df[['track_name', 'genre', 'genre_code']])
df.drop(columns=['genre'], inplace=True)
print(genre_mapping.to_string(index=False))


                        track_name        genre  genre_code
0                           Comedy     acoustic           0
1                 Ghost - Acoustic     acoustic           0
2                   To Begin Again     acoustic           0
3       Can't Help Falling In Love     acoustic           0
4                          Hold On     acoustic           0
...                            ...          ...         ...
113995         Sleep My Little Boy  world-music         114
113996            Water Into Light  world-music         114
113997              Miss Perfumado  world-music         114
113998                     Friends  world-music         114
113999                   Barbincor  world-music         114

[113999 rows x 3 columns]
       genre_name  genre_code
         acoustic           0
         afrobeat           1
         alt-rock           2
      alternative           3
          ambient           4
            anime           5
      black-metal           6
        blueg

In [5]:
df.head()
df.shape


(113999, 19)

In [6]:
df = df.drop_duplicates(subset=["track_name", "artists"])
df.reset_index(drop=True, inplace=True)
print(df.head())
print(df.shape)


                  artists                  track_name  popularity  \
0             Gen Hoshino                      Comedy          73   
1            Ben Woodward            Ghost - Acoustic          55   
2  Ingrid Michaelson;ZAYN              To Begin Again          57   
3            Kina Grannis  Can't Help Falling In Love          71   
4        Chord Overstreet                     Hold On          82   

   duration_ms  explicit  danceability  energy  key  loudness  mode  \
0       230666         0         0.676  0.4610    1    -6.746     0   
1       149610         0         0.420  0.1660    1   -17.235     1   
2       210826         0         0.438  0.3590    0    -9.734     1   
3       201933         0         0.266  0.0596    0   -18.515     1   
4       198853         0         0.618  0.4430    2    -9.681     1   

   speechiness  acousticness  instrumentalness  liveness  valence    tempo  \
0       0.1430        0.0322          0.000001    0.3580    0.715   87.917   
1 

In [168]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.spatial import distance


In [169]:

# returns the songs most similar to the one at the given index, with the caveat
# that it will only pick songs within the same genre.
def random_song_by_genre(song_name, song_artist):
  #
  
  song_index = -1
  find_index = df.loc[(df['track_name'] == song_name) & (df['artists'] == song_artist)]
  if find_index.empty:
    print("Oops, enter a valid artist and song instead!")
    return 
  else:
    found_index = df.loc[(df['track_name'] == song_name) & (df['artists'] == song_artist)].index
    song_index = found_index[0]
  #print(index_list)
  print(song_index)
  #song_index = index_list[0]

  nn = NearestNeighbors(metric='euclidean', n_neighbors=15)

  # tracks the genre for each song index.
  all_song_names = df["track_name"].to_list()
  all_song_artists = df["artists"].to_list()
  all_song_genres = df["genre_code"].to_list()
  all_song_valence = df["valence"].to_list()
  all_song_danceability = df["danceability"].to_list()

  #info for the individual song
  song_name = all_song_names[song_index]
  print(song_name)
  song_artist = all_song_artists[song_index]
  song_genre = all_song_genres[song_index]
  song_valence = all_song_valence[song_index]
  song_danceability = all_song_danceability[song_index]

  song_distances = []
  
  #iterates through all songs
  for s in range(len(all_song_names)):
    #checks to make sure that they aren't the same song.
    if all_song_names[s] != song_name or all_song_artists[s] != song_artist:
      #checks to make sure that they are within the same genre.
      if all_song_genres[s] == song_genre:
        # okay, here you calculate the distance in terms of genre, valence and danceability.
        current_song = (song_genre,song_valence,song_danceability)
        #print(current_song)
        compare_song = (all_song_genres[s],all_song_valence[s],all_song_danceability[s])
        #print(compare_song)
        song_dist = distance.euclidean(current_song,compare_song)
        #contains song name, artist and distance from the current song.
        song_info = [all_song_names[s],all_song_artists[s],song_dist]
        #print(song_dist)

        song_distances.append(song_info)
  
  song_distances.sort(key=lambda x: x[2])

  similar_songs = [s for s in song_distances[:15]]
  print(similar_songs)

  return
print()
print()
print("We are returning the songs that are most comparable to ", df["track_name"][3], "by", df["artists"][3])
print()
random_song_by_genre("Can't Help Falling In Love", "Kina Grannis")

print()
print()
print("We are returning the songs that are most comparable to I Ain't Worried by One Republic")
print()
random_song_by_genre("I Ain't Worried", "OneRepublic")

print()
print()
print()
print("We are returning the songs that are most comparable to One Last Kiss by Hikaru Utada")
print()
random_song_by_genre("One Last Kiss", "Hikaru Utada")
print()

print()
print()
print()
print("We are returning the songs that are most comparable to Piano Man by Billy Joel")
print()
random_song_by_genre("Piano Man", "Billy Joel")
print()

print()
print()
print()
print("We are returning the songs that are most comparable to You Belong With Me (Taylor’s Version) by Taylor Swift")
print()
random_song_by_genre("You Belong With Me (Taylor’s Version)", "Taylor Swift")
print()




We are returning the songs that are most comparable to  Can't Help Falling In Love by Kina Grannis

3
Can't Help Falling In Love
[["Can't Help Falling In Love - Piano Version", 'Kina Grannis', 0.006403124237432854], ['In the Morning', 'JJ Heller', 0.014212670403551892], ['Golden Hour - Acoustic Piano', 'Ben Woodward', 0.026925824035672518], ['Over You', 'Ingrid Michaelson;A Great Big World', 0.03342154993413679], ['The Power of Love', 'Gabrielle Aplin', 0.03383784863137727], ['Blackbird Song', 'Lee DeWyze', 0.04060788100849391], ['bouquet', 'Ichiko Aoba', 0.04252058325093857], ['To Whom It May Concern', 'The Civil Wars', 0.04318564576337835], ['Stars Are on Your Side', 'Ross Copperman', 0.049244289008980535], ["I Ain't Ever Loved No One - Acoustic", 'Donovan Woods;Tenille Townes', 0.058523499553598125], ['Poison & Wine', 'The Civil Wars', 0.06293647591023827], ['Pirates of the Caribbean Theme', 'Eddie van der Meer', 0.06389053137985314], ["Don't Know What I Want", 'Gabrielle Aplin', 